In [1]:

# ============================================================
# Burgers checkpoint KKT conditioning density sweep
# DAMPED SWEEP FIRST
#
# Uses one saved theta checkpoint:
#   theta_star_pointwise_jitter_1e-5.npz.npy
#
# For each PDE collocation density:
#   5x5, 10x10, 20x20, 30x30
#
# It assembles the constraint Jacobian J and KKT matrices:
#   K_no_damp = [[H, J.T],
#                [J, 0]]
#
#   K_damped  = [[H, J.T],
#                [J, -mu I]]
#
# and reports:
#   sigma_min(J)
#   numerical rank(J), with tol = rank_rtol * sigma_max(J)
#   cond(J J^T)
#   cond(K) without damping
#   cond(K) with damping
#
# This does NOT retrain.
# It only loads theta and computes diagnostics.
# ============================================================

import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import math
import csv
import time
from pathlib import Path

import numpy as np
import scipy.io
import scipy.linalg

import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

import jax.numpy as jnp
from jax import random, grad, vmap, hessian
from functools import partial


# ============================================================
# User settings
# ============================================================
THETA_PATH = "theta_star_pointwise_jitter_1e-5.npz.npy"
BURGERS_PATH = "burgers.mat"

OUT_CSV = "burgers_damped_checkpoint_density_sweep_kkt.csv"

# Architecture must match training
hidden_dim = 30
num_hidden = 3
layer_sizes = [2] + [hidden_dim] * num_hidden + [1]

DTYPE = jnp.float64
EPS = DTYPE(1e-12)

# Damped KKT block
MU_DAMP = 1e-4

# Numerical rank tolerance:
# rank_tol = RANK_RTOL * sigma_max(J)
RANK_RTOL = 1e-10

# Density sweep. These are PDE constraint grids.
PDE_DENSITIES = [
    (5, 5),
    (10, 10),
    (20, 20),
    (30, 30),
]

# Keep BC/IC same as your Burgers SQP code
K_BC = 100
K_IC = 200

# Jitter setting for diagnostic assembly.
# Set to 1e-5 to match your stochastic jitter scale.
# Set to 0.0 for deterministic base-point diagnostics.
USE_JITTER = True
JITTER_SIGMA_X = 1e-5
JITTER_SIGMA_T = 1e-5
JITTER_CLIP_K = 2.0

# Fixed seed makes the diagnostic reproducible.
SEED = 12345


# ============================================================
# Network utilities
# ============================================================
def mlp_apply(params, x):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h


def make_shapes(layer_sizes):
    shapes = []
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        shapes.append(((m, n), (n,)))
    return tuple(shapes)


def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)

        W = theta[idx: idx + W_size].reshape(W_shape)
        idx += W_size

        b = theta[idx: idx + b_size].reshape(b_shape)
        idx += b_size

        params.append({"W": W, "b": b})

    if idx != theta.size:
        raise ValueError(f"theta size mismatch: used {idx}, theta has {theta.size}")

    return params


def load_theta(path, shapes):
    arr = np.load(path)

    if isinstance(arr, np.lib.npyio.NpzFile):
        print("Loaded NPZ keys:", arr.files)
        arr = arr[arr.files[0]]

    theta_np = np.asarray(arr).reshape(-1)
    expected_n = sum(math.prod(Ws) + math.prod(bs) for Ws, bs in shapes)

    print("theta size:", theta_np.size)
    print("expected size:", expected_n)

    if theta_np.size != expected_n:
        raise ValueError(
            f"Wrong architecture or theta file. theta has {theta_np.size}, expected {expected_n}"
        )

    return jnp.asarray(theta_np, dtype=DTYPE)


# ============================================================
# Data/eval utilities
# ============================================================
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)

    t = d["t"].squeeze()
    x = d["x"].squeeze()
    usol = d["usol"]

    if usol.shape == (len(x), len(t)):
        usol = usol.T
    elif usol.shape != (len(t), len(x)):
        raise ValueError(
            f"Unexpected usol shape {usol.shape}; expected {(len(t), len(x))} or {(len(x), len(t))}"
        )

    nu = float(np.array(d["nu"]).squeeze()) if "nu" in d else 0.01 / np.pi
    return t, x, usol, nu


def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape


@partial(jax.jit, static_argnames=("shapes",))
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]


def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.asarray(X_grid_np, dtype=DTYPE)
    u_pred = np.asarray(predict_u(theta, shapes, X_grid)).reshape(grid_shape)

    mse = float(np.mean((u_pred - usol_np) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - usol_np) / (np.linalg.norm(usol_np) + 1e-12))
    return mse, rel_l2


# ============================================================
# Sampling utilities
# ============================================================
def sample_pde_stratified_density(key, nx, nt, x_min, x_max, t_min, t_max):
    K = nx * nt
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(nx)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(nt)

    it, ix = jnp.meshgrid(
        jnp.arange(nt),
        jnp.arange(nx),
        indexing="ij",
    )

    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0 + u[:, 0] * dx
    ts = t0 + u[:, 1] * dt

    return jnp.stack([xs, ts], axis=1)


def sample_bc_stratified(key, x_min, x_max, t_min, t_max):
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(K_BC)
    j = jnp.arange(K_BC, dtype=jnp.int32)
    t0 = DTYPE(t_min) + DTYPE(j) * dt

    u = random.uniform(key, (K_BC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (t0 + u * dt).reshape(-1, 1)

    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(ts), ts], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(ts), ts], axis=1)
    return XL, XR


def sample_ic_stratified(key, x_min, x_max, t0, x_grid, u0_grid):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(K_IC)
    j = jnp.arange(K_IC, dtype=jnp.int32)
    x0 = DTYPE(x_min) + DTYPE(j) * dx

    u = random.uniform(key, (K_IC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (x0 + u * dx).reshape(-1, 1)

    Xic = jnp.concatenate([xs, DTYPE(t0) * jnp.ones_like(xs)], axis=1)
    u0 = jnp.interp(xs[:, 0], x_grid, u0_grid)
    return Xic, u0


def jitter_points(key, X, x_min, x_max, t_min, t_max,
                  sigma_x, sigma_t, clip_k=2.0):
    if (sigma_x <= 0.0) and (sigma_t <= 0.0):
        return X

    key, kx, kt = random.split(key, 3)

    dx = DTYPE(sigma_x) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
    dt = DTYPE(sigma_t) * random.normal(kt, (X.shape[0],), dtype=DTYPE)

    if clip_k is not None:
        ck = DTYPE(clip_k)
        if sigma_x > 0:
            dx = jnp.clip(dx, -ck * DTYPE(sigma_x), ck * DTYPE(sigma_x))
        if sigma_t > 0:
            dt = jnp.clip(dt, -ck * DTYPE(sigma_t), ck * DTYPE(sigma_t))

    x = jnp.clip(X[:, 0] + dx, DTYPE(x_min), DTYPE(x_max))
    t = jnp.clip(X[:, 1] + dt, DTYPE(t_min), DTYPE(t_max))
    return jnp.stack([x, t], axis=1)


# ============================================================
# PDE residual and constraints
# ============================================================
def pde_residual_unscaled(params, X_f, nu):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X_f)
    du = vmap(grad(u_fun))(X_f)
    H = vmap(hessian(u_fun))(X_f)

    u_x = du[:, 0]
    u_t = du[:, 1]
    u_xx = H[:, 0, 0]
    return u_t + u * u_x - DTYPE(nu) * u_xx


@jax.jit
def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux


@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector_density(theta, shapes,
                              X_f,
                              X_bc_L, X_bc_R,
                              X_ic, u0_ic,
                              nu):
    params = unflatten_params(theta, shapes)

    # PDE residual constraints
    c_pde = pde_residual_unscaled(params, X_f, nu)

    # Periodic BC value and derivative constraints
    uL, uxL = u_and_ux(params, X_bc_L)
    uR, uxR = u_and_ux(params, X_bc_R)
    c_bc_val = uL - uR
    c_bc_der = uxL - uxR

    # IC constraints
    u_ic = mlp_apply(params, X_ic)[:, 0]
    c_ic = u_ic - u0_ic

    return jnp.concatenate([c_pde, c_bc_val, c_bc_der, c_ic], axis=0)


@partial(jax.jit, static_argnames=("shapes",))
def C_and_J_density(theta, shapes,
                    X_f,
                    X_bc_L, X_bc_R,
                    X_ic, u0_ic,
                    nu):
    def c_fun(th):
        return constraint_vector_density(
            th, shapes, X_f, X_bc_L, X_bc_R, X_ic, u0_ic, nu
        )

    def c_aux(th):
        c = c_fun(th)
        return c, c

    J, c = jax.jacrev(c_aux, has_aux=True)(theta)
    return c, J


# ============================================================
# Conditioning diagnostics
# ============================================================
def safe_cond(A):
    try:
        return float(np.linalg.cond(A))
    except Exception:
        return float("nan")


def kkt_matrix(H, J, mu):
    n = H.shape[0]
    m = J.shape[0]

    top = np.concatenate([H, J.T], axis=1)
    bottom = np.concatenate([J, -float(mu) * np.eye(m)], axis=1)
    return np.concatenate([top, bottom], axis=0)


def compute_diagnostics(J_np, mu_damp, rank_rtol):
    m, n = J_np.shape
    H_np = np.eye(n, dtype=np.float64)

    svals = scipy.linalg.svdvals(J_np)
    sigma_max = float(svals[0]) if svals.size else float("nan")
    sigma_min = float(svals[-1]) if svals.size else float("nan")

    rank_tol = float(rank_rtol * sigma_max)
    rank = int(np.sum(svals > rank_tol))

    JJt = J_np @ J_np.T
    cond_JJt = safe_cond(JJt)

    K0 = kkt_matrix(H_np, J_np, mu=0.0)
    Kd = kkt_matrix(H_np, J_np, mu=mu_damp)

    cond_K_no_damp = safe_cond(K0)
    cond_K_damped = safe_cond(Kd)

    return {
        "m_constraints": m,
        "n_params": n,
        "sigma_min_J": sigma_min,
        "sigma_max_J": sigma_max,
        "rank_J": rank,
        "rank_tol": rank_tol,
        "cond_JJt": cond_JJt,
        "cond_K_no_damp": cond_K_no_damp,
        "cond_K_damped": cond_K_damped,
    }


# ============================================================
# Main sweep
# ============================================================
def main():
    t0_all = time.time()

    shapes = make_shapes(layer_sizes)
    theta = load_theta(THETA_PATH, shapes)

    t_np, x_np, usol_np, nu = load_burgers_mat(BURGERS_PATH)
    x_min = float(x_np.min())
    x_max = float(x_np.max())
    t_min = float(t_np.min())
    t_max = float(t_np.max())

    x_grid = jnp.asarray(x_np, dtype=DTYPE)
    u0_grid = jnp.asarray(usol_np[0, :], dtype=DTYPE)

    mse, rel_l2 = eval_full_grid(theta, shapes, x_np, t_np, usol_np)
    print(f"[checkpoint eval] MSE={mse:.6e}, relL2={rel_l2:.6e}")

    key = random.PRNGKey(SEED)

    rows = []

    for nx, nt in PDE_DENSITIES:
        print("\n" + "=" * 80)
        print(f"Density {nx}x{nt} = {nx*nt} PDE constraints")
        t0 = time.time()

        key, k_pde, k_bc, k_ic = random.split(key, 4)

        X_f = sample_pde_stratified_density(k_pde, nx, nt, x_min, x_max, t_min, t_max)
        XL, XR = sample_bc_stratified(k_bc, x_min, x_max, t_min, t_max)
        Xic, u0ic = sample_ic_stratified(k_ic, x_min, x_max, t_min, x_grid, u0_grid)

        if USE_JITTER:
            key, kj_f, kj_bc, kj_ic = random.split(key, 4)

            X_f = jitter_points(
                kj_f, X_f, x_min, x_max, t_min, t_max,
                JITTER_SIGMA_X, JITTER_SIGMA_T, JITTER_CLIP_K
            )

            # BC: jitter only t, keep x exactly on boundaries.
            XL = jitter_points(
                kj_bc, XL, x_min, x_max, t_min, t_max,
                0.0, JITTER_SIGMA_T, JITTER_CLIP_K
            )
            XR = jitter_points(
                kj_bc, XR, x_min, x_max, t_min, t_max,
                0.0, JITTER_SIGMA_T, JITTER_CLIP_K
            )

            # IC: jitter only x, keep t = 0.
            Xic = jitter_points(
                kj_ic, Xic, x_min, x_max, t_min, t_max,
                JITTER_SIGMA_X, 0.0, JITTER_CLIP_K
            )
            u0ic = jnp.interp(Xic[:, 0], x_grid, u0_grid)

        c, J = C_and_J_density(
            theta, shapes,
            X_f,
            XL, XR,
            Xic, u0ic,
            DTYPE(nu),
        )

        c_np = np.asarray(c)
        J_np = np.asarray(J)

        diag = compute_diagnostics(J_np, MU_DAMP, RANK_RTOL)

        pde_slice = slice(0, nx * nt)
        bc_val_slice = slice(nx * nt, nx * nt + K_BC)
        bc_der_slice = slice(nx * nt + K_BC, nx * nt + 2 * K_BC)
        ic_slice = slice(nx * nt + 2 * K_BC, None)

        def rms(a):
            return float(np.sqrt(np.mean(a * a) + 1e-30))

        row = {
            "theta_path": THETA_PATH,
            "nu": nu,
            "mu_damp": MU_DAMP,
            "rank_rtol": RANK_RTOL,
            "use_jitter": USE_JITTER,
            "jitter_sigma_x": JITTER_SIGMA_X if USE_JITTER else 0.0,
            "jitter_sigma_t": JITTER_SIGMA_T if USE_JITTER else 0.0,
            "nx_pde": nx,
            "nt_pde": nt,
            "pde_points": nx * nt,
            "bc_value_points": K_BC,
            "bc_derivative_points": K_BC,
            "ic_points": K_IC,
            "pde_rms": rms(c_np[pde_slice]),
            "bc_val_rms": rms(c_np[bc_val_slice]),
            "bc_der_rms": rms(c_np[bc_der_slice]),
            "ic_rms": rms(c_np[ic_slice]),
            "checkpoint_mse": mse,
            "checkpoint_relL2": rel_l2,
            "elapsed_sec": time.time() - t0,
            **diag,
        }

        rows.append(row)

        print(f"sigma_min(J)     = {row['sigma_min_J']:.6e}")
        print(f"sigma_max(J)     = {row['sigma_max_J']:.6e}")
        print(f"rank(J)          = {row['rank_J']} / {row['m_constraints']}")
        print(f"rank_tol         = {row['rank_tol']:.6e}")
        print(f"cond(JJ^T)       = {row['cond_JJt']:.6e}")
        print(f"cond(K no damp)  = {row['cond_K_no_damp']:.6e}")
        print(f"cond(K damped)   = {row['cond_K_damped']:.6e}")
        print(f"elapsed          = {row['elapsed_sec']:.2f}s")

    fieldnames = list(rows[0].keys())
    with open(OUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("\nSaved:", OUT_CSV)
    print(f"Total elapsed: {time.time() - t0_all:.2f}s")


if __name__ == "__main__":
    main()


theta size: 1981
expected size: 1981
[checkpoint eval] MSE=1.441166e-07, relL2=6.188130e-04

Density 5x5 = 25 PDE constraints
sigma_min(J)     = 2.247270e-15
sigma_max(J)     = 2.839806e+02
rank(J)          = 151 / 425
rank_tol         = 2.839806e-08
cond(JJ^T)       = 1.123809e+20
cond(K no damp)  = 1.999733e+20
cond(K damped)   = 2.844810e+06
elapsed          = 73.44s

Density 10x10 = 100 PDE constraints
sigma_min(J)     = 2.303976e-15
sigma_max(J)     = 7.261986e+02
rank(J)          = 219 / 500
rank_tol         = 7.261986e-08
cond(JJ^T)       = 3.933389e+20
cond(K no damp)  = 6.742983e+20
cond(K damped)   = 7.266987e+06
elapsed          = 246.50s

Density 20x20 = 400 PDE constraints
sigma_min(J)     = 1.805662e-15
sigma_max(J)     = 5.137076e+02
rank(J)          = 509 / 800
rank_tol         = 5.137076e-08
cond(JJ^T)       = 1.436810e+20
cond(K no damp)  = 3.576356e+20
cond(K damped)   = 5.142078e+06
elapsed          = 82.96s

Density 30x30 = 900 PDE constraints
sigma_min(J)     = 1.

In [1]:

# ============================================================
# Burgers checkpoint KKT conditioning density sweep
# DAMPED SWEEP FIRST
#
# Uses four saved theta checkpoints:
#   theta_damped_diag_200_iter_1.npy
#   theta_damped_diag_200_iter_10.npy
#   theta_damped_diag_200_iter_100.npy
#   theta_damped_diag_200_iter_200.npy
#
# Uses the standard PDE collocation density:
#   30x30
#
# It assembles the constraint Jacobian J and KKT matrices:
#   K_no_damp = [[H, J.T],
#                [J, 0]]
#
#   K_damped  = [[H, J.T],
#                [J, -mu I]]
#
# and reports:
#   sigma_min(J)
#   numerical rank(J), with tol = rank_rtol * sigma_max(J)
#   cond(J J^T)
#   cond(K) without damping
#   cond(K) with damping
#
# This does NOT retrain.
# It only loads theta and computes diagnostics.
# ============================================================

import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import math
import csv
import time
from pathlib import Path

import numpy as np
import scipy.io
import scipy.linalg

import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

import jax.numpy as jnp
from jax import random, grad, vmap, hessian
from functools import partial


# ============================================================
# User settings
# ============================================================
# Checkpoint files saved from the 200-iteration damped SQP diagnostic run.
# These are the only theta files used for the professor's few-iterates table.
THETA_CHECKPOINTS = [
    ("iter_1", "theta_damped_diag_200_iter_1.npy"),
    ("iter_10", "theta_damped_diag_200_iter_10.npy"),
    ("iter_100", "theta_damped_diag_200_iter_100.npy"),
    ("iter_200", "theta_damped_diag_200_iter_200.npy"),
]
BURGERS_PATH = "burgers.mat"

OUT_CSV = "burgers_damped_200_checkpoint_kkt_diagnostics.csv"

# Architecture must match training
hidden_dim = 30
num_hidden = 3
layer_sizes = [2] + [hidden_dim] * num_hidden + [1]

DTYPE = jnp.float64
EPS = DTYPE(1e-12)

# Damped KKT block
MU_DAMP = 1e-4

# Numerical rank tolerance:
# rank_tol = RANK_RTOL * sigma_max(J)
RANK_RTOL = 1e-10

# For the professor's few-iterates table, use the standard training density.
# Density sweep is separate/optional.
CHECK_DENSITY = (30, 30)

# Keep BC/IC same as your Burgers SQP code
K_BC = 100
K_IC = 200

# Jitter setting for diagnostic assembly.
# Set to 1e-5 to match your stochastic jitter scale.
# Set to 0.0 for deterministic base-point diagnostics.
USE_JITTER = True
JITTER_SIGMA_X = 1e-5
JITTER_SIGMA_T = 1e-5
JITTER_CLIP_K = 2.0

# Fixed seed makes the diagnostic reproducible.
SEED = 12345


# ============================================================
# Network utilities
# ============================================================
def mlp_apply(params, x):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h


def make_shapes(layer_sizes):
    shapes = []
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        shapes.append(((m, n), (n,)))
    return tuple(shapes)


def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)

        W = theta[idx: idx + W_size].reshape(W_shape)
        idx += W_size

        b = theta[idx: idx + b_size].reshape(b_shape)
        idx += b_size

        params.append({"W": W, "b": b})

    if idx != theta.size:
        raise ValueError(f"theta size mismatch: used {idx}, theta has {theta.size}")

    return params


def load_theta(path, shapes):
    arr = np.load(path)

    if isinstance(arr, np.lib.npyio.NpzFile):
        print("Loaded NPZ keys:", arr.files)
        arr = arr[arr.files[0]]

    theta_np = np.asarray(arr).reshape(-1)
    expected_n = sum(math.prod(Ws) + math.prod(bs) for Ws, bs in shapes)

    print("theta size:", theta_np.size)
    print("expected size:", expected_n)

    if theta_np.size != expected_n:
        raise ValueError(
            f"Wrong architecture or theta file. theta has {theta_np.size}, expected {expected_n}"
        )

    return jnp.asarray(theta_np, dtype=DTYPE)


# ============================================================
# Data/eval utilities
# ============================================================
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)

    t = d["t"].squeeze()
    x = d["x"].squeeze()
    usol = d["usol"]

    if usol.shape == (len(x), len(t)):
        usol = usol.T
    elif usol.shape != (len(t), len(x)):
        raise ValueError(
            f"Unexpected usol shape {usol.shape}; expected {(len(t), len(x))} or {(len(x), len(t))}"
        )

    nu = float(np.array(d["nu"]).squeeze()) if "nu" in d else 0.01 / np.pi
    return t, x, usol, nu


def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape


@partial(jax.jit, static_argnames=("shapes",))
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]


def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.asarray(X_grid_np, dtype=DTYPE)
    u_pred = np.asarray(predict_u(theta, shapes, X_grid)).reshape(grid_shape)

    mse = float(np.mean((u_pred - usol_np) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - usol_np) / (np.linalg.norm(usol_np) + 1e-12))
    return mse, rel_l2


# ============================================================
# Sampling utilities
# ============================================================
def sample_pde_stratified_density(key, nx, nt, x_min, x_max, t_min, t_max):
    K = nx * nt
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(nx)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(nt)

    it, ix = jnp.meshgrid(
        jnp.arange(nt),
        jnp.arange(nx),
        indexing="ij",
    )

    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0 + u[:, 0] * dx
    ts = t0 + u[:, 1] * dt

    return jnp.stack([xs, ts], axis=1)


def sample_bc_stratified(key, x_min, x_max, t_min, t_max):
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(K_BC)
    j = jnp.arange(K_BC, dtype=jnp.int32)
    t0 = DTYPE(t_min) + DTYPE(j) * dt

    u = random.uniform(key, (K_BC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (t0 + u * dt).reshape(-1, 1)

    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(ts), ts], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(ts), ts], axis=1)
    return XL, XR


def sample_ic_stratified(key, x_min, x_max, t0, x_grid, u0_grid):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(K_IC)
    j = jnp.arange(K_IC, dtype=jnp.int32)
    x0 = DTYPE(x_min) + DTYPE(j) * dx

    u = random.uniform(key, (K_IC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (x0 + u * dx).reshape(-1, 1)

    Xic = jnp.concatenate([xs, DTYPE(t0) * jnp.ones_like(xs)], axis=1)
    u0 = jnp.interp(xs[:, 0], x_grid, u0_grid)
    return Xic, u0


def jitter_points(key, X, x_min, x_max, t_min, t_max,
                  sigma_x, sigma_t, clip_k=2.0):
    if (sigma_x <= 0.0) and (sigma_t <= 0.0):
        return X

    key, kx, kt = random.split(key, 3)

    dx = DTYPE(sigma_x) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
    dt = DTYPE(sigma_t) * random.normal(kt, (X.shape[0],), dtype=DTYPE)

    if clip_k is not None:
        ck = DTYPE(clip_k)
        if sigma_x > 0:
            dx = jnp.clip(dx, -ck * DTYPE(sigma_x), ck * DTYPE(sigma_x))
        if sigma_t > 0:
            dt = jnp.clip(dt, -ck * DTYPE(sigma_t), ck * DTYPE(sigma_t))

    x = jnp.clip(X[:, 0] + dx, DTYPE(x_min), DTYPE(x_max))
    t = jnp.clip(X[:, 1] + dt, DTYPE(t_min), DTYPE(t_max))
    return jnp.stack([x, t], axis=1)


# ============================================================
# PDE residual and constraints
# ============================================================
def pde_residual_unscaled(params, X_f, nu):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X_f)
    du = vmap(grad(u_fun))(X_f)
    H = vmap(hessian(u_fun))(X_f)

    u_x = du[:, 0]
    u_t = du[:, 1]
    u_xx = H[:, 0, 0]
    return u_t + u * u_x - DTYPE(nu) * u_xx


@jax.jit
def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux


@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector_density(theta, shapes,
                              X_f,
                              X_bc_L, X_bc_R,
                              X_ic, u0_ic,
                              nu):
    params = unflatten_params(theta, shapes)

    # PDE residual constraints
    c_pde = pde_residual_unscaled(params, X_f, nu)

    # Periodic BC value and derivative constraints
    uL, uxL = u_and_ux(params, X_bc_L)
    uR, uxR = u_and_ux(params, X_bc_R)
    c_bc_val = uL - uR
    c_bc_der = uxL - uxR

    # IC constraints
    u_ic = mlp_apply(params, X_ic)[:, 0]
    c_ic = u_ic - u0_ic

    return jnp.concatenate([c_pde, c_bc_val, c_bc_der, c_ic], axis=0)


@partial(jax.jit, static_argnames=("shapes",))
def C_and_J_density(theta, shapes,
                    X_f,
                    X_bc_L, X_bc_R,
                    X_ic, u0_ic,
                    nu):
    def c_fun(th):
        return constraint_vector_density(
            th, shapes, X_f, X_bc_L, X_bc_R, X_ic, u0_ic, nu
        )

    def c_aux(th):
        c = c_fun(th)
        return c, c

    J, c = jax.jacrev(c_aux, has_aux=True)(theta)
    return c, J


# ============================================================
# Conditioning diagnostics
# ============================================================
def safe_cond(A):
    try:
        return float(np.linalg.cond(A))
    except Exception:
        return float("nan")


def kkt_matrix(H, J, mu):
    n = H.shape[0]
    m = J.shape[0]

    top = np.concatenate([H, J.T], axis=1)
    bottom = np.concatenate([J, -float(mu) * np.eye(m)], axis=1)
    return np.concatenate([top, bottom], axis=0)


def compute_diagnostics(J_np, mu_damp, rank_rtol):
    m, n = J_np.shape
    H_np = np.eye(n, dtype=np.float64)

    svals = scipy.linalg.svdvals(J_np)
    sigma_max = float(svals[0]) if svals.size else float("nan")
    sigma_min = float(svals[-1]) if svals.size else float("nan")

    rank_tol = float(rank_rtol * sigma_max)
    rank = int(np.sum(svals > rank_tol))

    JJt = J_np @ J_np.T
    cond_JJt = safe_cond(JJt)

    K0 = kkt_matrix(H_np, J_np, mu=0.0)
    Kd = kkt_matrix(H_np, J_np, mu=mu_damp)

    cond_K_no_damp = safe_cond(K0)
    cond_K_damped = safe_cond(Kd)

    return {
        "m_constraints": m,
        "n_params": n,
        "sigma_min_J": sigma_min,
        "sigma_max_J": sigma_max,
        "rank_J": rank,
        "rank_tol": rank_tol,
        "cond_JJt": cond_JJt,
        "cond_K_no_damp": cond_K_no_damp,
        "cond_K_damped": cond_K_damped,
    }


# ============================================================
# Main checkpoint-iteration diagnostics
# ============================================================
def main():
    t0_all = time.time()

    shapes = make_shapes(layer_sizes)

    t_np, x_np, usol_np, nu = load_burgers_mat(BURGERS_PATH)
    x_min = float(x_np.min())
    x_max = float(x_np.max())
    t_min = float(t_np.min())
    t_max = float(t_np.max())

    x_grid = jnp.asarray(x_np, dtype=DTYPE)
    u0_grid = jnp.asarray(usol_np[0, :], dtype=DTYPE)

    key = random.PRNGKey(SEED)
    rows = []

    # For professor's "few iterates" table:
    # use the normal 30x30 constraint density for every checkpoint.
    nx, nt = CHECK_DENSITY
    print(f"Using one diagnostic density: {nx}x{nt} = {nx*nt} PDE constraints")
    print("Checkpoints:")
    for label, path in THETA_CHECKPOINTS:
        print(f"  {label}: {path}")

    for label, theta_path in THETA_CHECKPOINTS:
        print("\n" + "=" * 80)
        print(f"Checkpoint {label}: {theta_path}")
        t0 = time.time()

        if not Path(theta_path).exists():
            print(f"WARNING: missing checkpoint file: {theta_path}")
            continue

        theta = load_theta(theta_path, shapes)
        mse, rel_l2 = eval_full_grid(theta, shapes, x_np, t_np, usol_np)
        print(f"[checkpoint eval] MSE={mse:.6e}, relL2={rel_l2:.6e}")

        # Use a checkpoint-specific split so every checkpoint is evaluated on
        # a reproducible but identical type of collocation realization.
        key, k_pde, k_bc, k_ic = random.split(key, 4)

        X_f = sample_pde_stratified_density(k_pde, nx, nt, x_min, x_max, t_min, t_max)
        XL, XR = sample_bc_stratified(k_bc, x_min, x_max, t_min, t_max)
        Xic, u0ic = sample_ic_stratified(k_ic, x_min, x_max, t_min, x_grid, u0_grid)

        if USE_JITTER:
            key, kj_f, kj_bc, kj_ic = random.split(key, 4)

            X_f = jitter_points(
                kj_f, X_f, x_min, x_max, t_min, t_max,
                JITTER_SIGMA_X, JITTER_SIGMA_T, JITTER_CLIP_K
            )

            # BC: jitter only t, keep x exactly on boundaries.
            XL = jitter_points(
                kj_bc, XL, x_min, x_max, t_min, t_max,
                0.0, JITTER_SIGMA_T, JITTER_CLIP_K
            )
            XR = jitter_points(
                kj_bc, XR, x_min, x_max, t_min, t_max,
                0.0, JITTER_SIGMA_T, JITTER_CLIP_K
            )

            # IC: jitter only x, keep t = 0.
            Xic = jitter_points(
                kj_ic, Xic, x_min, x_max, t_min, t_max,
                JITTER_SIGMA_X, 0.0, JITTER_CLIP_K
            )
            u0ic = jnp.interp(Xic[:, 0], x_grid, u0_grid)

        c, J = C_and_J_density(
            theta, shapes,
            X_f,
            XL, XR,
            Xic, u0ic,
            DTYPE(nu),
        )

        c_np = np.asarray(c)
        J_np = np.asarray(J)
        diag = compute_diagnostics(J_np, MU_DAMP, RANK_RTOL)

        pde_slice = slice(0, nx * nt)
        bc_val_slice = slice(nx * nt, nx * nt + K_BC)
        bc_der_slice = slice(nx * nt + K_BC, nx * nt + 2 * K_BC)
        ic_slice = slice(nx * nt + 2 * K_BC, None)

        def rms_np(a):
            return float(np.sqrt(np.mean(a * a) + 1e-30))

        row = {
            "checkpoint": label,
            "theta_path": theta_path,
            "nu": nu,
            "mu_damp": MU_DAMP,
            "rank_rtol": RANK_RTOL,
            "use_jitter": USE_JITTER,
            "jitter_sigma_x": JITTER_SIGMA_X if USE_JITTER else 0.0,
            "jitter_sigma_t": JITTER_SIGMA_T if USE_JITTER else 0.0,
            "nx_pde": nx,
            "nt_pde": nt,
            "pde_points": nx * nt,
            "bc_value_points": K_BC,
            "bc_derivative_points": K_BC,
            "ic_points": K_IC,
            "pde_rms": rms_np(c_np[pde_slice]),
            "bc_val_rms": rms_np(c_np[bc_val_slice]),
            "bc_der_rms": rms_np(c_np[bc_der_slice]),
            "ic_rms": rms_np(c_np[ic_slice]),
            "checkpoint_mse": mse,
            "checkpoint_relL2": rel_l2,
            "elapsed_sec": time.time() - t0,
            **diag,
        }

        rows.append(row)

        print(f"sigma_min(J)     = {row['sigma_min_J']:.6e}")
        print(f"sigma_max(J)     = {row['sigma_max_J']:.6e}")
        print(f"rank(J)          = {row['rank_J']} / {row['m_constraints']}")
        print(f"rank_tol         = {row['rank_tol']:.6e}")
        print(f"cond(JJ^T)       = {row['cond_JJt']:.6e}")
        print(f"cond(K no damp)  = {row['cond_K_no_damp']:.6e}")
        print(f"cond(K damped)   = {row['cond_K_damped']:.6e}")
        print(f"elapsed          = {row['elapsed_sec']:.2f}s")

    if not rows:
        raise RuntimeError("No rows were computed. Check checkpoint filenames and paths.")

    fieldnames = list(rows[0].keys())
    with open(OUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("\nSaved:", OUT_CSV)
    print(f"Total elapsed: {time.time() - t0_all:.2f}s")


if __name__ == "__main__":
    main()


Using one diagnostic density: 30x30 = 900 PDE constraints
Checkpoints:
  iter_1: theta_damped_diag_200_iter_1.npy
  iter_10: theta_damped_diag_200_iter_10.npy
  iter_100: theta_damped_diag_200_iter_100.npy
  iter_200: theta_damped_diag_200_iter_200.npy

Checkpoint iter_1: theta_damped_diag_200_iter_1.npy
theta size: 1981
expected size: 1981
[checkpoint eval] MSE=3.382457e-01, relL2=9.480225e-01
sigma_min(J)     = 1.304816e-16
sigma_max(J)     = 1.881957e+02
rank(J)          = 350 / 1300
rank_tol         = 1.881957e-08
cond(JJ^T)       = 5.992408e+20
cond(K no damp)  = 1.211110e+22
cond(K damped)   = 1.886963e+06
elapsed          = 22.98s

Checkpoint iter_10: theta_damped_diag_200_iter_10.npy
theta size: 1981
expected size: 1981
[checkpoint eval] MSE=3.006267e-01, relL2=8.937505e-01
sigma_min(J)     = 2.427208e-16
sigma_max(J)     = 2.473337e+02
rank(J)          = 383 / 1300
rank_tol         = 2.473337e-08
cond(JJ^T)       = 1.848028e+20
cond(K no damp)  = 7.449319e+20
cond(K damped)   

In [1]:
# ============================================================
# Burgers checkpoint-by-density KKT conditioning diagnostics
# ITERATION x DENSITY SWEEP
#
# This combines your two current scripts:
#   1) checkpoint table at one density
#   2) final/best checkpoint density sweep
#
# It evaluates every checkpoint on every PDE density:
#   checkpoints: iter_1, iter_10, iter_100, iter_200, final_best
#   densities:   5x5, 10x10, 20x20, 30x30
#
# For each pair, it assembles the constraint Jacobian J and KKT matrices:
#   K_no_damp = [[H, J.T],
#                [J, 0]]
#
#   K_damped  = [[H, J.T],
#                [J, -mu I]]
#
# and reports:
#   sigma_min(J)
#   numerical rank(J), with tol = rank_rtol * sigma_max(J)
#   cond(J J^T)
#   cond(K) without damping
#   cond(K) with damping
#
# This does NOT retrain.
# It only loads saved theta checkpoints and computes diagnostics.
#
# Important fairness choice:
#   For a given density, the same diagnostic collocation points are reused
#   across all checkpoints. BC/IC diagnostic points are also fixed across
#   all densities/checkpoints. This makes changes reflect the checkpoint
#   and/or PDE density, not different random diagnostic points.
# ============================================================

import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import math
import csv
import time
from pathlib import Path

import numpy as np
import scipy.io
import scipy.linalg

import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_matmul_precision", "highest")

import jax.numpy as jnp
from jax import random, grad, vmap, hessian
from functools import partial



# ============================================================
# User settings
# ============================================================
# Checkpoints saved from the damped SQP diagnostic run + optional final/best theta.
# If final_best is missing, the script skips it with a warning.
THETA_CHECKPOINTS = [
    ("iter_1", "theta_damped_diag_200_iter_1.npy"),
    ("iter_10", "theta_damped_diag_200_iter_10.npy"),
    ("iter_100", "theta_damped_diag_200_iter_100.npy"),
    ("iter_200", "theta_damped_diag_200_iter_200.npy"),
    ("final_best", "theta_star_pointwise_jitter_1e-5.npz.npy"),
]

BURGERS_PATH = "burgers.mat"

OUT_CSV = "burgers_checkpoint_by_density_kkt_diagnostics.csv"

# Architecture must match training
hidden_dim = 30
num_hidden = 3
layer_sizes = [2] + [hidden_dim] * num_hidden + [1]

DTYPE = jnp.float64
EPS = DTYPE(1e-12)

# Damped KKT block
MU_DAMP = 1e-4

# Numerical rank tolerance:
# rank_tol = RANK_RTOL * sigma_max(J)
RANK_RTOL = 1e-10

# PDE density sweep. These are PDE constraint grids.
PDE_DENSITIES = [
    (5, 5),
    (10, 10),
    (20, 20),
    (30, 30),
]

# Keep BC/IC same as your Burgers SQP code
K_BC = 100
K_IC = 200

# Jitter setting for diagnostic assembly.
# Set to 1e-5 to match your stochastic jitter scale.
# Set to 0.0 for deterministic base-point diagnostics.
USE_JITTER = True
JITTER_SIGMA_X = 1e-5
JITTER_SIGMA_T = 1e-5
JITTER_CLIP_K = 2.0

# Fixed seed makes the diagnostic reproducible.
SEED = 12345

# ============================================================
# Network utilities
# ============================================================
def mlp_apply(params, x):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h


def make_shapes(layer_sizes):
    shapes = []
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        shapes.append(((m, n), (n,)))
    return tuple(shapes)


def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)

        W = theta[idx: idx + W_size].reshape(W_shape)
        idx += W_size

        b = theta[idx: idx + b_size].reshape(b_shape)
        idx += b_size

        params.append({"W": W, "b": b})

    if idx != theta.size:
        raise ValueError(f"theta size mismatch: used {idx}, theta has {theta.size}")

    return params


def load_theta(path, shapes):
    arr = np.load(path)

    if isinstance(arr, np.lib.npyio.NpzFile):
        print("Loaded NPZ keys:", arr.files)
        arr = arr[arr.files[0]]

    theta_np = np.asarray(arr).reshape(-1)
    expected_n = sum(math.prod(Ws) + math.prod(bs) for Ws, bs in shapes)

    print("theta size:", theta_np.size)
    print("expected size:", expected_n)

    if theta_np.size != expected_n:
        raise ValueError(
            f"Wrong architecture or theta file. theta has {theta_np.size}, expected {expected_n}"
        )

    return jnp.asarray(theta_np, dtype=DTYPE)


# ============================================================
# Data/eval utilities
# ============================================================
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)

    t = d["t"].squeeze()
    x = d["x"].squeeze()
    usol = d["usol"]

    if usol.shape == (len(x), len(t)):
        usol = usol.T
    elif usol.shape != (len(t), len(x)):
        raise ValueError(
            f"Unexpected usol shape {usol.shape}; expected {(len(t), len(x))} or {(len(x), len(t))}"
        )

    nu = float(np.array(d["nu"]).squeeze()) if "nu" in d else 0.01 / np.pi
    return t, x, usol, nu


def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape


@partial(jax.jit, static_argnames=("shapes",))
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]


def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.asarray(X_grid_np, dtype=DTYPE)
    u_pred = np.asarray(predict_u(theta, shapes, X_grid)).reshape(grid_shape)

    mse = float(np.mean((u_pred - usol_np) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - usol_np) / (np.linalg.norm(usol_np) + 1e-12))
    return mse, rel_l2


# ============================================================
# Sampling utilities
# ============================================================
def sample_pde_stratified_density(key, nx, nt, x_min, x_max, t_min, t_max):
    K = nx * nt
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(nx)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(nt)

    it, ix = jnp.meshgrid(
        jnp.arange(nt),
        jnp.arange(nx),
        indexing="ij",
    )

    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0 + u[:, 0] * dx
    ts = t0 + u[:, 1] * dt

    return jnp.stack([xs, ts], axis=1)


def sample_bc_stratified(key, x_min, x_max, t_min, t_max):
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(K_BC)
    j = jnp.arange(K_BC, dtype=jnp.int32)
    t0 = DTYPE(t_min) + DTYPE(j) * dt

    u = random.uniform(key, (K_BC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (t0 + u * dt).reshape(-1, 1)

    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(ts), ts], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(ts), ts], axis=1)
    return XL, XR


def sample_ic_stratified(key, x_min, x_max, t0, x_grid, u0_grid):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(K_IC)
    j = jnp.arange(K_IC, dtype=jnp.int32)
    x0 = DTYPE(x_min) + DTYPE(j) * dx

    u = random.uniform(key, (K_IC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (x0 + u * dx).reshape(-1, 1)

    Xic = jnp.concatenate([xs, DTYPE(t0) * jnp.ones_like(xs)], axis=1)
    u0 = jnp.interp(xs[:, 0], x_grid, u0_grid)
    return Xic, u0


def jitter_points(key, X, x_min, x_max, t_min, t_max,
                  sigma_x, sigma_t, clip_k=2.0):
    if (sigma_x <= 0.0) and (sigma_t <= 0.0):
        return X

    key, kx, kt = random.split(key, 3)

    dx = DTYPE(sigma_x) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
    dt = DTYPE(sigma_t) * random.normal(kt, (X.shape[0],), dtype=DTYPE)

    if clip_k is not None:
        ck = DTYPE(clip_k)
        if sigma_x > 0:
            dx = jnp.clip(dx, -ck * DTYPE(sigma_x), ck * DTYPE(sigma_x))
        if sigma_t > 0:
            dt = jnp.clip(dt, -ck * DTYPE(sigma_t), ck * DTYPE(sigma_t))

    x = jnp.clip(X[:, 0] + dx, DTYPE(x_min), DTYPE(x_max))
    t = jnp.clip(X[:, 1] + dt, DTYPE(t_min), DTYPE(t_max))
    return jnp.stack([x, t], axis=1)


# ============================================================
# PDE residual and constraints
# ============================================================
def pde_residual_unscaled(params, X_f, nu):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X_f)
    du = vmap(grad(u_fun))(X_f)
    H = vmap(hessian(u_fun))(X_f)

    u_x = du[:, 0]
    u_t = du[:, 1]
    u_xx = H[:, 0, 0]
    return u_t + u * u_x - DTYPE(nu) * u_xx


@jax.jit
def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux


@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector_density(theta, shapes,
                              X_f,
                              X_bc_L, X_bc_R,
                              X_ic, u0_ic,
                              nu):
    params = unflatten_params(theta, shapes)

    # PDE residual constraints
    c_pde = pde_residual_unscaled(params, X_f, nu)

    # Periodic BC value and derivative constraints
    uL, uxL = u_and_ux(params, X_bc_L)
    uR, uxR = u_and_ux(params, X_bc_R)
    c_bc_val = uL - uR
    c_bc_der = uxL - uxR

    # IC constraints
    u_ic = mlp_apply(params, X_ic)[:, 0]
    c_ic = u_ic - u0_ic

    return jnp.concatenate([c_pde, c_bc_val, c_bc_der, c_ic], axis=0)


@partial(jax.jit, static_argnames=("shapes",))
def C_and_J_density(theta, shapes,
                    X_f,
                    X_bc_L, X_bc_R,
                    X_ic, u0_ic,
                    nu):
    def c_fun(th):
        return constraint_vector_density(
            th, shapes, X_f, X_bc_L, X_bc_R, X_ic, u0_ic, nu
        )

    def c_aux(th):
        c = c_fun(th)
        return c, c

    J, c = jax.jacrev(c_aux, has_aux=True)(theta)
    return c, J


# ============================================================
# Conditioning diagnostics
# ============================================================
def safe_cond(A):
    try:
        return float(np.linalg.cond(A))
    except Exception:
        return float("nan")


def kkt_matrix(H, J, mu):
    n = H.shape[0]
    m = J.shape[0]

    top = np.concatenate([H, J.T], axis=1)
    bottom = np.concatenate([J, -float(mu) * np.eye(m)], axis=1)
    return np.concatenate([top, bottom], axis=0)


def compute_diagnostics(J_np, mu_damp, rank_rtol):
    m, n = J_np.shape
    H_np = np.eye(n, dtype=np.float64)

    svals = scipy.linalg.svdvals(J_np)
    sigma_max = float(svals[0]) if svals.size else float("nan")
    sigma_min = float(svals[-1]) if svals.size else float("nan")

    rank_tol = float(rank_rtol * sigma_max)
    rank = int(np.sum(svals > rank_tol))

    JJt = J_np @ J_np.T
    cond_JJt = safe_cond(JJt)

    K0 = kkt_matrix(H_np, J_np, mu=0.0)
    Kd = kkt_matrix(H_np, J_np, mu=mu_damp)

    cond_K_no_damp = safe_cond(K0)
    cond_K_damped = safe_cond(Kd)

    return {
        "m_constraints": m,
        "n_params": n,
        "sigma_min_J": sigma_min,
        "sigma_max_J": sigma_max,
        "rank_J": rank,
        "rank_tol": rank_tol,
        "cond_JJt": cond_JJt,
        "cond_K_no_damp": cond_K_no_damp,
        "cond_K_damped": cond_K_damped,
    }



# ============================================================
# Diagnostic point construction
# ============================================================
def build_diagnostic_sets(x_min, x_max, t_min, t_max, x_grid, u0_grid):
    """
    Build fixed diagnostic collocation sets.

    For fairness:
      - PDE points are fixed for each density.
      - BC/IC points are fixed and reused for all densities.
      - The same points are reused across all checkpoints.
      - If jitter is enabled, jitter is applied once here and then fixed.
    """
    key = random.PRNGKey(SEED)

    # Common BC/IC points for all densities and checkpoints.
    key, k_bc, k_ic = random.split(key, 3)
    XL_common, XR_common = sample_bc_stratified(k_bc, x_min, x_max, t_min, t_max)
    Xic_common, u0ic_common = sample_ic_stratified(k_ic, x_min, x_max, t_min, x_grid, u0_grid)

    if USE_JITTER:
        key, kj_bc, kj_ic = random.split(key, 3)

        # BC: jitter only t, keep x exactly on boundaries.
        # Use the same jitter key for left/right so periodic pairs stay paired in time.
        XL_common = jitter_points(
            kj_bc, XL_common, x_min, x_max, t_min, t_max,
            0.0, JITTER_SIGMA_T, JITTER_CLIP_K
        )
        XR_common = jitter_points(
            kj_bc, XR_common, x_min, x_max, t_min, t_max,
            0.0, JITTER_SIGMA_T, JITTER_CLIP_K
        )

        # IC: jitter only x, keep t = 0.
        Xic_common = jitter_points(
            kj_ic, Xic_common, x_min, x_max, t_min, t_max,
            JITTER_SIGMA_X, 0.0, JITTER_CLIP_K
        )
        u0ic_common = jnp.interp(Xic_common[:, 0], x_grid, u0_grid)

    diag_sets = {}

    for nx, nt in PDE_DENSITIES:
        key, k_pde = random.split(key, 2)

        X_f = sample_pde_stratified_density(k_pde, nx, nt, x_min, x_max, t_min, t_max)

        if USE_JITTER:
            key, kj_f = random.split(key, 2)
            X_f = jitter_points(
                kj_f, X_f, x_min, x_max, t_min, t_max,
                JITTER_SIGMA_X, JITTER_SIGMA_T, JITTER_CLIP_K
            )

        diag_sets[(nx, nt)] = {
            "X_f": X_f,
            "XL": XL_common,
            "XR": XR_common,
            "Xic": Xic_common,
            "u0ic": u0ic_common,
        }

    return diag_sets


# ============================================================
# Main checkpoint x density diagnostics
# ============================================================
def main():
    t0_all = time.time()

    shapes = make_shapes(layer_sizes)

    t_np, x_np, usol_np, nu = load_burgers_mat(BURGERS_PATH)
    x_min = float(x_np.min())
    x_max = float(x_np.max())
    t_min = float(t_np.min())
    t_max = float(t_np.max())

    x_grid = jnp.asarray(x_np, dtype=DTYPE)
    u0_grid = jnp.asarray(usol_np[0, :], dtype=DTYPE)

    diag_sets = build_diagnostic_sets(
        x_min, x_max, t_min, t_max,
        x_grid, u0_grid,
    )

    rows = []

    print("Checkpoint-by-density KKT diagnostics")
    print("Densities:")
    for nx, nt in PDE_DENSITIES:
        print(f"  {nx}x{nt} = {nx*nt} PDE constraints")
    print("Checkpoints:")
    for label, path in THETA_CHECKPOINTS:
        print(f"  {label}: {path}")
    print("BC/IC diagnostic points are shared across all densities/checkpoints.")
    print("PDE diagnostic points are shared across all checkpoints for each density.")

    for checkpoint_order, (label, theta_path) in enumerate(THETA_CHECKPOINTS):
        print("\n" + "#" * 90)
        print(f"Checkpoint {label}: {theta_path}")

        if not Path(theta_path).exists():
            print(f"WARNING: missing checkpoint file: {theta_path}; skipping.")
            continue

        theta = load_theta(theta_path, shapes)
        checkpoint_mse, checkpoint_rel_l2 = eval_full_grid(theta, shapes, x_np, t_np, usol_np)
        print(f"[checkpoint eval] MSE={checkpoint_mse:.6e}, relL2={checkpoint_rel_l2:.6e}")

        for density_order, (nx, nt) in enumerate(PDE_DENSITIES):
            print("\n" + "=" * 80)
            print(f"Checkpoint {label}, density {nx}x{nt} = {nx*nt} PDE constraints")
            t0 = time.time()

            ds = diag_sets[(nx, nt)]
            X_f = ds["X_f"]
            XL = ds["XL"]
            XR = ds["XR"]
            Xic = ds["Xic"]
            u0ic = ds["u0ic"]

            c, J = C_and_J_density(
                theta, shapes,
                X_f,
                XL, XR,
                Xic, u0ic,
                DTYPE(nu),
            )

            c_np = np.asarray(c)
            J_np = np.asarray(J)
            diag = compute_diagnostics(J_np, MU_DAMP, RANK_RTOL)

            pde_slice = slice(0, nx * nt)
            bc_val_slice = slice(nx * nt, nx * nt + K_BC)
            bc_der_slice = slice(nx * nt + K_BC, nx * nt + 2 * K_BC)
            ic_slice = slice(nx * nt + 2 * K_BC, None)

            def rms_np(a):
                return float(np.sqrt(np.mean(a * a) + 1e-30))

            row = {
                "checkpoint_order": checkpoint_order,
                "checkpoint": label,
                "theta_path": theta_path,
                "density_order": density_order,
                "nu": nu,
                "mu_damp": MU_DAMP,
                "rank_rtol": RANK_RTOL,
                "use_jitter": USE_JITTER,
                "jitter_sigma_x": JITTER_SIGMA_X if USE_JITTER else 0.0,
                "jitter_sigma_t": JITTER_SIGMA_T if USE_JITTER else 0.0,
                "same_points_across_checkpoints": True,
                "same_bc_ic_across_densities": True,
                "nx_pde": nx,
                "nt_pde": nt,
                "pde_points": nx * nt,
                "bc_value_points": K_BC,
                "bc_derivative_points": K_BC,
                "ic_points": K_IC,
                "pde_rms": rms_np(c_np[pde_slice]),
                "bc_val_rms": rms_np(c_np[bc_val_slice]),
                "bc_der_rms": rms_np(c_np[bc_der_slice]),
                "ic_rms": rms_np(c_np[ic_slice]),
                "checkpoint_mse": checkpoint_mse,
                "checkpoint_relL2": checkpoint_rel_l2,
                "elapsed_sec": time.time() - t0,
                **diag,
            }

            rows.append(row)

            print(f"sigma_min(J)     = {row['sigma_min_J']:.6e}")
            print(f"sigma_max(J)     = {row['sigma_max_J']:.6e}")
            print(f"rank(J)          = {row['rank_J']} / {row['m_constraints']}")
            print(f"rank_tol         = {row['rank_tol']:.6e}")
            print(f"cond(JJ^T)       = {row['cond_JJt']:.6e}")
            print(f"cond(K no damp)  = {row['cond_K_no_damp']:.6e}")
            print(f"cond(K damped)   = {row['cond_K_damped']:.6e}")
            print(f"elapsed          = {row['elapsed_sec']:.2f}s")

    if not rows:
        raise RuntimeError("No rows were computed. Check checkpoint filenames and paths.")

    fieldnames = list(rows[0].keys())
    with open(OUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print("\nSaved:", OUT_CSV)
    print(f"Rows written: {len(rows)}")
    print(f"Total elapsed: {time.time() - t0_all:.2f}s")


if __name__ == "__main__":
    main()


Checkpoint-by-density KKT diagnostics
Densities:
  5x5 = 25 PDE constraints
  10x10 = 100 PDE constraints
  20x20 = 400 PDE constraints
  30x30 = 900 PDE constraints
Checkpoints:
  iter_1: theta_damped_diag_200_iter_1.npy
  iter_10: theta_damped_diag_200_iter_10.npy
  iter_100: theta_damped_diag_200_iter_100.npy
  iter_200: theta_damped_diag_200_iter_200.npy
  final_best: theta_star_pointwise_jitter_1e-5.npz.npy
BC/IC diagnostic points are shared across all densities/checkpoints.
PDE diagnostic points are shared across all checkpoints for each density.

##########################################################################################
Checkpoint iter_1: theta_damped_diag_200_iter_1.npy
theta size: 1981
expected size: 1981
[checkpoint eval] MSE=3.382457e-01, relL2=9.480225e-01

Checkpoint iter_1, density 5x5 = 25 PDE constraints
sigma_min(J)     = 3.172398e-16
sigma_max(J)     = 1.013639e+02
rank(J)          = 88 / 425
rank_tol         = 1.013639e-08
cond(JJ^T)       = 5.084165e